<a href="https://colab.research.google.com/github/Madmins07/Madmins07/blob/main/Kdag_Task1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ***Problem Statement***

> As part of a survey aimed at understanding students' career aspirations, you've gathered
responses from students about their career dreams, but there's a hitch—loads of spelling
mistakes! For instance, "DOCTOR" is misspelled as "DACTAR" or "DOCTER," and
"ENGINEER" as "ENJINEAR."
To clean up the data and get accurate insights, you need a smart algorithm to fix these spelling
goof-ups. Your task is to develop an algorithm that can automatically correct these and makes
your form submission free from any spelling mistakes.

List of career options in the form are as follows :
1. Doctor
2. Lawyer
3. Teacher
4. Engineer
5. Accountant
6. Nurse
7. Police
8. Architect
9. Dentist
10. Pharmacist

> __IMPORTANT NOTE__



1.   Making sure that the excel sheet and the notebook are being operated in the same working directory.

2.   Using a modified version of the excel sheet provided, the only modification made is addition of the column name and no test data has been changed. Either one can add a new column to the top of the file and name the field as misspelled_word or one can use this data sheet:- https://drive.google.com/drive/folders/1ihs5XNXcLBvdeIYxAz8MJix6Ywf7a5Su

# ***Importing Libaries and reading the submission data***

> ### **Importing all the Libraries**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


 > ### **Reading the submission data**




In [ ]:
given_words = pd.read_excel("tassk1.xlsx")

In [ ]:
#Function to print the data stored in the excel file
def print_data(given_words,count):
    print("Data stored in the file is as follows:-\n")
    for index, row in given_words.iterrows():
        misspelled_word = row["misspelled_word"]
        print(f"{index+1}) {misspelled_word}\n")
        count+=1
    return count
count=print_data(given_words,0)

In [ ]:
print("Total number of submissions:- ",count)


> ### **Data Set Initialization**

In [ ]:
#Intialising data-set with correct words and a dictionary to store count of respective words
correct_words = ['Doctor', 'Engineer', 'Architect', 'Police', 'Teacher','Accountant','Nurse','Dentist','Pharmacist','Lawyer']
dict={'Doctor':0, 'Engineer':0, 'Architect':0, 'Police':0, 'Teacher':0,'Accountant':0,'Nurse':0,'Dentist':0,'Pharmacist':0,'Lawyer':0}

# ***Levenshtein Distance***

> The Levenshtein distance is a measure of the minimum number of single-character edits (insertions, deletions or substitutions) required to change one word into the other. Using it here as metric for measuring the similarity between two strings.

In [ ]:
def ld(str1, str2):

    # Getting the lengths of the input strings
    size_x = len(str1) + 1
    size_y = len(str2) + 1

    # Creating a ans matrix to store the Levenshtein distances
    # The ans matrix has shape:- size_x rows and size_y columns
    ans = np.zeros ((size_x, size_y))

    # Initializing the first row and column of the  ans

    for x in range(size_x):
        ans [x, 0] = x  # The distance from an empty string to a string of length x is x
    for y in range(size_y):
        ans [0, y] = y  # The distance from an empty string to a string of length y is y

    # Filling in the rest of the ans matrix by iterating over the two strings
    for x in range(1, size_x):
        for y in range(1, size_y):
            if (str1[x-1] == str2[y-1]): # If the characters of the two strings str1 and str2 are similar

                '''Taking minimum of the distances:
                1. Deleting the current character in str1 (ans[x-1, y] + 1)
                2. Matching the current characters (ans[x-1, y-1])
                3. Inserting the current character in str2 (ans[x, y-1] + 1)'''

                ans [x,y] = min(ans[x-1, y] + 1,ans[x-1, y-1] ,ans[x, y-1] + 1)
            else:

                '''If the characters are different
                The distance is the minimum of:
                1. Deleting the current character in str1 (matrix[x-1, y] + 1)
                2. Substituting the current characters (matrix[x-1, y-1] + 1)
                3. Inserting the current character in str2 (matrix[x, y-1] + 1)'''

                ans[x,y] = min(ans[x-1, y] + 1,ans[x-1, y-1] + 1,ans[x, y-1] + 1)

    # Returning the Levenshtein distance which is stored in the bottom-right corner of the matrix
    return  ans[size_x - 1, size_y - 1]

# ***Levenshtein Similarity***

> The similarity function calculates a score between 0 and 1 indicating how similar two strings are. A score of 1 means the strings are identical, while a score of 0 means they are completely dissimilar. Higher the similarity score, similar the two string are wrt each other.



In [ ]:
#Function to normalize levenshtein distance values
def ls(str1, str2):
    dist = ld(str1, str2) # Function call to find the levenshtein distance between the two strings
    # Getting the length of the two string
    len1 = len(str1)
    len2 = len(str2)
    return 1 - (dist / max(len1, len2)) # Nomalising the distance value by dividing it with the maximum of the two lengths

# ***Counter Function, Main function solve, functions to print graph and corrected data***

In [ ]:
#Counter function to get count of different words in the misspelled_words data set
def count_word(word,dict):
    if word in dict:
        dict[word] += 1
    else:
        dict[word] = 1
    return dict

In [ ]:
#Function to iterate over every misspelled word in the data set and print the respective correct word
def solve(given_words):
    words = []  # Creating an empty list to store the corrected words

    for index, row in given_words.iterrows(): # Using a row iterator to traverse over the excel file

        misspelled_word = row["misspelled_word"] # Storing the misspelled word

        # Initialising an array to store levenshtein similarity values for every misspelled word wrt every correct word in the data set
        ans=np.zeros((10,))

        pos=0 # Variable to traverse the index of the array

        # Iterating over every word in correct_words and storing the levenshtein similarity value in the ans array
        for correct_word in correct_words:
            if(pos<=9):
                ans[pos]=ls(correct_word,misspelled_word.lower())
            else:
                break
            pos+=1 # Incrementing position value

        words.append(correct_words[ans.argmax()]) # Appending the corrected word

        '''Finding the index of the highest similarity value in the ans array
        and getting the correct word wrt to the index returned'''

        count_word(correct_words[ans.argmax()],dict) # Passing the correct word to increase its count in the dictionary
        print(f"{index+1} ) The misspelled word:- {misspelled_word}, The corrected word is:- {correct_words[ans.argmax()]}\n")

    # Creating a new excel file to store the corrected data as well!

    base = pd.DataFrame(words, columns=['Words'])
    base.to_excel('words.xlsx', index=False)


In [ ]:
#Function to display the respective counts of career choices
def print_count(dict):
    print("Count of different career choices are as follows:-\n")
    for idx,i in enumerate(dict.keys()): #Enumerating so as to retain count of loops while traversing the dictionary
        print(f"{idx+1}) {i} -> {dict[i]}")


In [ ]:
#Function to print career choices with their respective frequencies
def print_graph(dict):
    words = list(dict.keys()) # Storing different careers as a list(words)
    counts = list(dict.values()) # Storing frequencies of respective careers as a list(counts)
    plt.figure(figsize=(10, 5))
    plt.xlabel('Career Aspirations',alpha=0.7)
    plt.ylabel('Count/Frequency',alpha=0.7)
    plt.title('Career Choices')
    plt.tight_layout()
    plt.bar(words, counts) # Creating a bar graph to be plotted
    plt.show()


# ***Correcting the data, Plotting bar graph of career aspirations, Creating a new excel file to store the data***

In [ ]:
solve(given_words) #Calling our main function so as to correct the data set of misspelled words

In [ ]:
print_count(dict) # Printing respective frequencies of each career

In [ ]:
print_graph(dict) # Displaying the bar graph

> Creating another excel file to store the newly correct words

In [ ]:
words=pd.read_excel("words.xlsx") # Gets stroed in the current working directory for later access

In [ ]:
#Function to print the data stored in the newly created excel file
def print_new_data(words,count):
    print("Printing the newly corrected data:-\n")
    for index, row in words.iterrows():
        word = row["Words"]
        print(f"{index+1}) {word}\n")
        count+=1
    return count

In [ ]:
count=print_new_data(words,0)
print("\nTotal number of newly corrected submissions data:- ",count)